In [1]:
import sys 
from bpemb import BPEmb
from tqdm import tqdm
import numpy as np
import os
import torch
from gensim.models import KeyedVectors
from collections import defaultdict

sys.path.append('../datasets')
sys.path.append("..")

## Create BP embedding for MUSE (SINGLE language).

In [3]:
def read(file, threshold=0, vocabulary=None, dtype='float'):
    header = file.readline().split(' ')
    count = int(header[0]) if threshold <= 0 else min(threshold, int(header[0]))
    dim = int(header[1])
    words = []
    matrix = np.empty((count, dim),  dtype=dtype) if vocabulary is None else []
    for i in tqdm(range(count)):
        word, vec = file.readline().split(' ', 1)
        if vocabulary is None:
            words.append(word)
            matrix[i] = np.fromstring(vec, sep=' ',  dtype=dtype)
        elif word in vocabulary:
            words.append(word)
            matrix.append(np.fromstring(vec, sep=' ',  dtype=dtype))
    return (words, matrix) if vocabulary is None else (words, torch.tensor(matrix,  dtype=dtype))

def get_dict(dict_path, source, target):

    dictf = open(dict_path, encoding='utf-8', errors='surrogateescape')
    src2trg = defaultdict(set)

    vocab = set()

    for line in dictf:
        splitted = line.split()
        if len(splitted) > 2:
            # Only using first translation if many are provided
            src, trg = splitted[:2]
        elif len(splitted) == 2:
            src, trg = splitted

        src_ind = source.key_to_index[src]
        trg_ind = target.key_to_index[trg]
        src2trg[src_ind].add(trg_ind)
        vocab.add(src)
    return vocab, src2trg

In [3]:
data_path = '../datasets'

dataset_name = 'muse'
lang = 'en'
emb_dim_source = 100  
emb_dim_target = 50    
vs             = 200000

path_source = os.path.join(data_path, f'muse/embeddings/wiki.multi.{lang}.vec')
path_target = os.path.join(data_path, f'muse/embeddings/wiki.multi.{lang}.vec')

model_source = open(path_source, encoding='utf-8', errors='surrogateescape')
model_target = open(path_target, encoding='utf-8', errors='surrogateescape')

i2w_source, vectors_source = read(model_source)
i2w_target, vectors_target = read(model_target)

bpemb_source = BPEmb(lang=lang, dim=emb_dim_source, vs=vs)
bpemb_target = BPEmb(lang=lang, dim=emb_dim_target, vs=vs)

model_new_source = KeyedVectors(vector_size=emb_dim_source)
model_new_target = KeyedVectors(vector_size=emb_dim_target)

non_valid_indices = set()
valid_indices = set(list(range(len(vectors_source))))

model_new_source.vectors = np.zeros((vectors_source.shape[0], emb_dim_source))
model_new_target.vectors = np.zeros((vectors_target.shape[0], emb_dim_target))

for ix, word in enumerate(tqdm(i2w_source)):  
    word_embed_source = bpemb_source.embed(word)
    word_embed_target = bpemb_target.embed(word)

    if np.isnan(word_embed_source).any() or np.isnan(word_embed_target).any():
        print(word)
        non_valid_indices.add(ix)
        
    if word_embed_source.shape[0] != 1 or word_embed_target.shape[0] != 1:
        non_valid_indices.add(ix)
        continue 

    model_new_source.vectors[ix] = word_embed_source[:]
    model_new_target.vectors[ix] = word_embed_target[:]
    
   
valid_indices = valid_indices - non_valid_indices
print('Number of non valid indices:', len(non_valid_indices))
print('Number of valid indices:', len(valid_indices))

valid_indices = list(valid_indices)

model_new_source.vectors =  model_new_source.vectors[valid_indices]
model_new_target.vectors =  model_new_target.vectors[valid_indices]

model_new_source.index_to_key = [i2w_source[ix] for ix in valid_indices]
model_new_target.index_to_key = [i2w_target[ix] for ix in valid_indices]

model_new_source.key_to_index = {word:i for i, word in enumerate(model_new_source.index_to_key)}
model_new_target.key_to_index = {word:i for i, word in enumerate(model_new_target.index_to_key)}

assert len(model_new_source.vectors) == len(model_new_source.index_to_key)
assert len(model_new_source.vectors) == len(model_new_source.key_to_index.keys())
assert np.isnan(model_new_source.vectors).sum() == 0

assert len(model_new_target.vectors) == len(model_new_target.index_to_key)
assert len(model_new_target.vectors) == len(model_new_target.key_to_index.keys())
assert np.isnan(model_new_target.vectors).sum() == 0
assert len(model_new_source.vectors) == len(model_new_target.vectors)

model_new_source.save(f'../datasets/{dataset_name}_{lang}_BP_{emb_dim_source}_{vs//1000}K.d2v')
model_new_target.save(f'../datasets/{dataset_name}_{lang}_BP_{emb_dim_target}_{vs//1000}K.d2v')

100%|████████████████████████████████████████████████████████████████████████| 200000/200000 [00:16<00:00, 12253.08it/s]


Number of non valid indices: 80832
Number of valid indices: 119168


## Create BP embedding for twitter/wiki-gigaword.

In [3]:
data_path = '../datasets'
dataset_name = 'twitter'

emb_dim_source = 100  
emb_dim_target = 50   
vs             = 200000

path_source = f'../datasets/{dataset_name}_glove_{emb_dim_source}.d2v'
path_target = f'../datasets/{dataset_name}_glove_{emb_dim_target}.d2v'

model_source = KeyedVectors.load(path_source)
model_target = KeyedVectors.load(path_target)

bpemb_source = BPEmb(lang='en', dim=emb_dim_source, vs=vs)
bpemb_target = BPEmb(lang='en', dim=emb_dim_target, vs=vs)

model_new_source = KeyedVectors(vector_size=emb_dim_source)
model_new_target = KeyedVectors(vector_size=emb_dim_target)

model_new_source.vectors = np.zeros((model_source.vectors.shape[0], emb_dim_source))
model_new_target.vectors = np.zeros((model_target.vectors.shape[0], emb_dim_target))

print(model_new_source.vectors.shape)
non_valid_indices = set()
valid_indices = set(list(range(len(model_source.vectors))))

for ix, word in enumerate(tqdm(model_source.index_to_key)):  
    word_embed_source = bpemb_source.embed(word)
    word_embed_target = bpemb_target.embed(word)

    if word_embed_source.shape[0] != 1 or word_embed_target.shape[0] != 1:
        non_valid_indices.add(ix)
        continue
    
    if np.isnan(word_embed_source).any() or np.isnan(word_embed_target).any():
        print(word)
        non_valid_indices.add(ix)

    model_new_source.vectors[ix] = word_embed_source[:]
    model_new_target.vectors[ix] = word_embed_target[:]
    
   
valid_indices = valid_indices - non_valid_indices
print('Number of non valid indices:', len(non_valid_indices))
print('Number of valid indices:', len(valid_indices))

valid_indices = list(valid_indices)

model_new_source.vectors =  model_new_source.vectors[valid_indices]
model_new_target.vectors =  model_new_target.vectors[valid_indices]

model_new_source.index_to_key = [model_source.index_to_key[ix] for ix in valid_indices]
model_new_target.index_to_key = [model_target.index_to_key[ix] for ix in valid_indices]

model_new_source.key_to_index = {word:i for i, word in enumerate(model_new_source.index_to_key)}
model_new_target.key_to_index = {word:i for i, word in enumerate(model_new_target.index_to_key)}

assert len(model_new_source.vectors) == len(model_new_source.index_to_key)
assert len(model_new_source.vectors) == len(model_new_source.key_to_index.keys())
assert np.isnan(model_new_source.vectors).sum() == 0

assert len(model_new_target.vectors) == len(model_new_target.index_to_key)
assert len(model_new_target.vectors) == len(model_new_target.key_to_index.keys())
assert np.isnan(model_new_target.vectors).sum() == 0
assert len(model_new_source.vectors) == len(model_new_target.vectors)

model_new_source.save(f'../datasets/{dataset_name}_BP_{emb_dim_source}_{vs//1000}K.d2v')
model_new_target.save(f'../datasets/{dataset_name}_BP_{emb_dim_target}_{vs//1000}K.d2v')

(1193514, 100)


100%|██████████████████████████████████████████████████████████████████████| 1193514/1193514 [00:48<00:00, 24611.75it/s]


Number of non valid indices: 1101177
Number of valid indices: 92337


## Create BP embeddings for MUSE (Different languages - same dimensions)

In [4]:
data_path = '../datasets'

source_lang = 'en'
target_lang = 'fr'

emb_dim_source  = 100
emb_dim_target  = 100
vs              = 200000

source_path = os.path.join(data_path, f'muse/embeddings/wiki.multi.{source_lang}.vec')
target_path = os.path.join(data_path, f'muse/embeddings/wiki.multi.{target_lang}.vec')
vocab_path  = os.path.join(data_path, f'muse/dictionaries/{source_lang}-{target_lang}.txt') 

source_model = open(source_path, encoding='utf-8', errors='surrogateescape')
target_model = open(target_path, encoding='utf-8', errors='surrogateescape')

i2w_source, vectors_source = read(source_model)
i2w_target, vectors_target = read(target_model)

model_new_source = KeyedVectors(vector_size=emb_dim_source)
model_new_target = KeyedVectors(vector_size=emb_dim_target)

model_new_source.index_to_key = i2w_source[:]
model_new_target.index_to_key = i2w_target[:]

model_new_source.key_to_index = {word: i for i, word in enumerate(i2w_source)}
model_new_target.key_to_index = {word: i for i, word in enumerate(i2w_target)}

100%|████████████████████████████████████████████████████████████████████████| 200000/200000 [00:08<00:00, 22314.52it/s]


In [4]:
vocab, src2trg = get_dict(vocab_path, model_new_source, model_new_target)

valid_keys = []
for k in src2trg.keys():
    if src2trg[k] != set():
        valid_keys.append(k)

print(len(valid_keys))

indices_source = valid_keys[:]
indices_target = [min(src2trg[ix]) for ix in indices_source]

words_source = [model_new_source.index_to_key[ix] for ix in indices_source]
words_target = [model_new_target.index_to_key[ix] for ix in indices_target]

model_new_source.vectors = np.zeros((len(words_source), emb_dim_source))
model_new_target.vectors = np.zeros((len(words_target), emb_dim_target))

model_new_source.index_to_key = words_source
model_new_target.index_to_key = words_target

bpemb_source = BPEmb(lang=source_lang, dim=emb_dim_source, vs=vs)
bpemb_target = BPEmb(lang=target_lang, dim=emb_dim_target, vs=vs)

non_valid_indices = set()
valid_indices = set(list(range(len(model_new_source.vectors))))

for ix in tqdm(range(len(model_new_source.index_to_key))):#(tqdm(i2w_source)):  
    word_source       = model_new_source.index_to_key[ix]
    word_target       = model_new_target.index_to_key[ix]
    
    word_embed_source = bpemb_source.embed(word_source)
    word_embed_target = bpemb_target.embed(word_target)

    if np.isnan(word_embed_source).any() or np.isnan(word_embed_target).any():
        print(word)
        non_valid_indices.add(ix)
        
    if word_embed_source.shape[0] != 1 or word_embed_target.shape[0] != 1:
        non_valid_indices.add(ix)
        continue 

    model_new_source.vectors[ix] = word_embed_source[:]
    model_new_target.vectors[ix] = word_embed_target[:]
    
   
valid_indices = valid_indices - non_valid_indices
print('Number of non valid indices:', len(non_valid_indices))
print('Number of valid indices:', len(valid_indices))

valid_indices = list(valid_indices)

model_new_source.vectors =  model_new_source.vectors[valid_indices]
model_new_target.vectors =  model_new_target.vectors[valid_indices]

model_new_source.index_to_key = [i2w_source[ix] for ix in valid_indices]
model_new_target.index_to_key = [i2w_target[ix] for ix in valid_indices]

#model_new_source.key_to_index = {word:i for i, word in enumerate(model_new_source.index_to_key)}
#model_new_target.key_to_index = {word:i for i, word in enumerate(model_new_target.index_to_key)}

assert len(model_new_source.vectors) == len(model_new_source.index_to_key)
#assert len(model_new_source.vectors) == len(model_new_source.key_to_index.keys())
assert np.isnan(model_new_source.vectors).sum() == 0

assert len(model_new_target.vectors) == len(model_new_target.index_to_key)
#assert len(model_new_target.vectors) == len(model_new_target.key_to_index.keys())
assert np.isnan(model_new_target.vectors).sum() == 0
assert len(model_new_source.vectors) == len(model_new_target.vectors)

#model_new_source.save(f'../datasets/{dataset_name}_{lang}_BP_{emb_dim_source}_{vs//1000}K.d2v')
#model_new_target.save(f'../datasets/{dataset_name}_{lang}_BP_{emb_dim_target}_{vs//1000}K.d2v')

93084


100%|██████████████████████████████████████████████████████████████████████████| 93084/93084 [00:05<00:00, 18023.85it/s]


Number of non valid indices: 31315
Number of valid indices: 61769


In [ ]:
model_source_target = KeyedVectors(vector_size=emb_dim_source)
full_len = len(model_new_source.vectors)

if emb_dim_source == emb_dim_target:
    model_source_target.vectors = np.zeros((2*full_len, emb_dim_source), dtype=np.float32)

    for ix in range(2*full_len):
        if ix < full_len:
            model_source_target.vectors[ix] = model_new_source.vectors[ix]
            model_source_target.index_to_key.append(model_new_source.index_to_key[ix])
            
        else:
            model_source_target.vectors[ix] = model_new_target.vectors[ix-full_len]
            model_source_target.index_to_key.append(model_new_target.index_to_key[ix-full_len])
    
    model_source_target.save(f'../datasets/muse_{source_lang}(BP)({emb_dim_source})_{target_lang}(BP)({emb_dim_target}).d2v')

## Extracting fastText embeddings from MUSE and saving into KeyedVectors

In [7]:
data_path = '../datasets'

source_lang = 'en'
target_lang = 'fr'

emb_dim_source  = 300
emb_dim_target  = 300
vs              = 200000

source_path = os.path.join(data_path, f'muse/embeddings/wiki.multi.{source_lang}.vec')
target_path = os.path.join(data_path, f'muse/embeddings/wiki.multi.{target_lang}.vec')
vocab_path  = os.path.join(data_path, f'muse/dictionaries/{source_lang}-{target_lang}.txt') 

source_model = open(source_path, encoding='utf-8', errors='surrogateescape')
target_model = open(target_path, encoding='utf-8', errors='surrogateescape')

i2w_source, vectors_source = read(source_model)
i2w_target, vectors_target = read(target_model)

model_new_source = KeyedVectors(vector_size=emb_dim_source)
model_new_target = KeyedVectors(vector_size=emb_dim_target)

model_new_source.index_to_key = i2w_source[:]
model_new_target.index_to_key = i2w_target[:]

model_new_source.key_to_index = {word: i for i, word in enumerate(i2w_source)}
model_new_target.key_to_index = {word: i for i, word in enumerate(i2w_target)}

vocab, src2trg = get_dict(vocab_path, model_new_source, model_new_target)

valid_keys = []
for k in src2trg.keys():
    if src2trg[k] != set():
        valid_keys.append(k)

print(len(valid_keys))

indices_source = valid_keys[:]
indices_target = [min(src2trg[ix]) for ix in indices_source]

words_source = [model_new_source.index_to_key[ix] for ix in indices_source]
words_target = [model_new_target.index_to_key[ix] for ix in indices_target]

model_new_source.vectors = vectors_source[indices_source]
model_new_target.vectors = vectors_target[indices_target]

model_new_source.index_to_key = words_source
model_new_target.index_to_key = words_target

100%|████████████████████████████████████████████████████████████████████████| 200000/200000 [00:09<00:00, 21911.19it/s]


94681


In [8]:
model_source_target = KeyedVectors(vector_size=emb_dim_source)
full_len = len(model_new_source.vectors)

if emb_dim_source == emb_dim_target:
    model_source_target.vectors = np.zeros((2*full_len, emb_dim_source), dtype=np.float32)

    for ix in range(2*full_len):
        if ix < full_len:
            model_source_target.vectors[ix] = model_new_source.vectors[ix]
            model_source_target.index_to_key.append(model_new_source.index_to_key[ix])
            
        else:
            model_source_target.vectors[ix] = model_new_target.vectors[ix-full_len]
            model_source_target.index_to_key.append(model_new_target.index_to_key[ix-full_len])
    
    model_source_target.save(f'../datasets/muse_{source_lang}(fT)({emb_dim_source})_{target_lang}(fT)({emb_dim_target}).d2v')